# Notebook 3: ARIMA / SARIMAX Modelling
Fits a SARIMAX model per product, evaluates on 6-month holdout.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns_colors = ['steelblue', 'coral', 'seagreen', 'orchid', 'goldenrod', 'tomato', 'teal', 'slategray']

df_agg = pd.read_csv('../data/processed/featured_aggregated_data.csv', parse_dates=['date'])
CUTOFF = pd.Timestamp('2024-06-01')

def mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / actual)) * 100

products = df_agg['product_name'].unique()
print('Products to model:', products.tolist())

## 1. Fit SARIMAX per Product

In [ ]:
# SARIMAX(p,d,q)(P,D,Q,s) — s=12 for monthly data
# Order (1,1,1)(1,1,0,12) is a good general starting point for monthly pharma demand
ORDER = (1, 1, 1)
SEASONAL_ORDER = (1, 1, 0, 12)

arima_results = {}
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, prod in enumerate(products):
    series = df_agg[df_agg['product_name'] == prod].set_index('date')['total_units_sold'].asfreq('MS')
    train = series[series.index < CUTOFF]
    test  = series[series.index >= CUTOFF]

    model = SARIMAX(train, order=ORDER, seasonal_order=SEASONAL_ORDER,
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)

    forecast = fit.get_forecast(steps=len(test))
    pred_mean = forecast.predicted_mean
    conf_int  = forecast.conf_int(alpha=0.2)  # 80% CI

    mae_val  = mean_absolute_error(test, pred_mean)
    rmse_val = np.sqrt(mean_squared_error(test, pred_mean))
    mape_val = mape(test.values, pred_mean.values)

    arima_results[prod] = {'MAE': round(mae_val, 1), 'RMSE': round(rmse_val, 1), 'MAPE': round(mape_val, 2)}

    # Plot
    ax = axes[i]
    ax.plot(train.index, train.values, color='steelblue', label='Train', linewidth=1.5)
    ax.plot(test.index, test.values, color='green', label='Actual', linewidth=2, marker='o', markersize=4)
    ax.plot(pred_mean.index, pred_mean.values, color='tomato', label='SARIMAX Forecast', linewidth=2, linestyle='--')
    ax.fill_between(conf_int.index, conf_int.iloc[:,0], conf_int.iloc[:,1], alpha=0.15, color='tomato')
    ax.axvline(CUTOFF, color='gray', linestyle=':', linewidth=1)
    ax.set_title(f'{prod}  |  MAPE: {mape_val:.1f}%', fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    print(f'{prod:<22} MAE={mae_val:.0f}  RMSE={rmse_val:.0f}  MAPE={mape_val:.2f}%')

plt.suptitle('SARIMAX(1,1,1)(1,1,0,12) Forecasts — All Products', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/10_arima_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. ARIMA Evaluation Summary

In [ ]:
results_df = pd.DataFrame(arima_results).T.reset_index().rename(columns={'index': 'Product'})
results_df = results_df.sort_values('MAPE')
print('=== SARIMAX Evaluation on 6-Month Holdout ===')
print(results_df.to_string(index=False))
print(f"\nAverage MAPE across all products: {results_df['MAPE'].mean():.2f}%")
results_df.to_csv('../data/processed/arima_metrics.csv', index=False)

## 3. Future Forecast — Next 6 Months (2025)

In [ ]:
# Refit on full data and forecast Jan–Jun 2025
future_forecasts = {}
FUTURE_STEPS = 6

fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, prod in enumerate(products):
    full_series = df_agg[df_agg['product_name'] == prod].set_index('date')['total_units_sold'].asfreq('MS')
    model = SARIMAX(full_series, order=ORDER, seasonal_order=SEASONAL_ORDER,
                    enforce_stationarity=False, enforce_invertibility=False)
    fit = model.fit(disp=False)
    forecast = fit.get_forecast(steps=FUTURE_STEPS)
    pred = forecast.predicted_mean
    ci   = forecast.conf_int(alpha=0.2)
    future_forecasts[prod] = pred.round(0).astype(int).to_dict()

    ax = axes[i]
    ax.plot(full_series.index[-18:], full_series.values[-18:], color='steelblue', label='Historical', linewidth=2)
    ax.plot(pred.index, pred.values, color='tomato', label='Forecast (Jan–Jun 2025)', linewidth=2.5, linestyle='--', marker='o')
    ax.fill_between(ci.index, ci.iloc[:,0], ci.iloc[:,1], alpha=0.15, color='tomato')
    ax.set_title(prod, fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Future Demand Forecast: Jan–Jun 2025 (SARIMAX)', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/11_future_forecasts_arima.png', dpi=150, bbox_inches='tight')
plt.show()

# Export
pd.DataFrame(future_forecasts).T.to_csv('../data/processed/arima_future_forecast.csv')
print('Future forecasts exported.')